# 🚚 Supply Chain Late Delivery Prediction

## ML-Powered Logistics Intelligence | Case Study Presentation

---

<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%); padding: 30px; border-radius: 15px; color: white; margin: 20px 0;">

### 📋 Presentation Agenda

| Section | Topic | Time |
|---------|-------|------|
| **1** | Business Problem & Data Overview | 2 min |
| **2** | Exploratory Data Analysis | 2 min |
| **3** | Feature Engineering Deep Dive | 2 min |
| **4** | Modeling Approach & Results | 2 min |
| **5** | SHAP Interpretability & Business Impact | 2 min |

</div>

<div style="background: linear-gradient(135deg, #0D1117 0%, #161B22 100%); padding: 15px 25px; border-radius: 10px; border-left: 4px solid #00D4FF; margin: 15px 0;">

### 🔄 Automated Report Integration

This notebook **automatically reads the latest model performance report** from the `reports/` folder. All metrics, model comparisons, and feature importance data are dynamically parsed and visualized.

**To update results:** Simply re-run the training pipeline to generate a new report, then re-run this notebook.

</div>

---

**Presenter:** Data Science & ML Engineering  
**Duration:** 10 Minutes  
**Focus:** Intersection of Software Engineering & Data Science


In [44]:
# ============================================================
# 🔧 SETUP - All imports and configuration
# ============================================================
import sys
import warnings
warnings.filterwarnings('ignore')
sys.path.append('..')

# Core libraries
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from datetime import datetime

# Sklearn
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve, precision_recall_curve,
    confusion_matrix, classification_report
)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Models
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# SHAP
import shap
shap.initjs()
import matplotlib.pyplot as plt
%matplotlib inline

# ============================================================
# 🎨 DARK MODE COLOR PALETTE
# ============================================================
DARK_COLORS = {
    'background': '#0D1117',
    'card': '#161B22',
    'surface': '#21262D',
    'text': '#F0F6FC',
    'text_secondary': '#8B949E',
    'cyan': '#00D4FF',
    'magenta': '#FF006E',
    'lime': '#7CFC00',
    'amber': '#FFB000',
    'violet': '#8B5CF6',
    'teal': '#14B8A6',
    'coral': '#FF6B6B',
    'gold': '#FFD700',
}

# Categorical color palette (dark-mode friendly, high contrast)
CATEGORICAL_COLORS = [
    '#00D4FF',  # Cyan
    '#FF006E',  # Magenta
    '#7CFC00',  # Lime
    '#FFB000',  # Amber
    '#8B5CF6',  # Violet
    '#14B8A6',  # Teal
    '#FF6B6B',  # Coral
    '#FFD700',  # Gold
]

# Status colors for late delivery
STATUS_COLORS = {
    'late': '#FF006E',
    'ontime': '#7CFC00',
    'risk_high': '#FF6B6B',
    'risk_med': '#FFB000',
    'risk_low': '#14B8A6',
}

# Apply dark template to Plotly
dark_template = go.layout.Template()
dark_template.layout = go.Layout(
    paper_bgcolor=DARK_COLORS['background'],
    plot_bgcolor=DARK_COLORS['card'],
    font=dict(color=DARK_COLORS['text'], family='Inter, sans-serif'),
    title=dict(font=dict(size=18, color=DARK_COLORS['text']), x=0.5),
    xaxis=dict(gridcolor=DARK_COLORS['surface'], linecolor=DARK_COLORS['surface'],
               tickfont=dict(color=DARK_COLORS['text_secondary'])),
    yaxis=dict(gridcolor=DARK_COLORS['surface'], linecolor=DARK_COLORS['surface'],
               tickfont=dict(color=DARK_COLORS['text_secondary'])),
    legend=dict(bgcolor='rgba(0,0,0,0)', font=dict(color=DARK_COLORS['text'])),
    colorway=CATEGORICAL_COLORS
)
pio.templates['dark_presentation'] = dark_template
pio.templates.default = 'dark_presentation'

# Matplotlib dark theme for SHAP
plt.rcParams.update({
    'figure.facecolor': DARK_COLORS['background'],
    'axes.facecolor': DARK_COLORS['card'],
    'axes.edgecolor': DARK_COLORS['surface'],
    'axes.labelcolor': DARK_COLORS['text'],
    'text.color': DARK_COLORS['text'],
    'xtick.color': DARK_COLORS['text_secondary'],
    'ytick.color': DARK_COLORS['text_secondary'],
    'grid.color': DARK_COLORS['surface'],
})

# Configuration
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✅ All libraries loaded successfully")
print("🎨 Dark mode color palette applied")
print(f"📅 Presentation Date: {datetime.now().strftime('%B %d, %Y')}")

# ============================================================
# 🔍 AUTO-DETECT AVAILABLE REPORTS
# ============================================================
reports_dir = Path('../reports')
report_files = sorted(reports_dir.glob('model_performance_report*.md')) if reports_dir.exists() else []

if report_files:
    latest_report = report_files[-1]
    # Extract timestamp from filename
    import re
    timestamp_match = re.search(r'(\d{8}_\d{4})', latest_report.name)
    if timestamp_match:
        ts = timestamp_match.group(1)
        report_date = f"{ts[:4]}-{ts[4:6]}-{ts[6:8]} {ts[9:11]}:{ts[11:13]}"
    else:
        report_date = "Unknown"

    print(f"\n{'='*60}")
    print("📊 AUTO-LOAD CONFIGURATION")
    print(f"{'='*60}")
    print(f"   📁 Report File: {latest_report.name}")
    print(f"   📅 Report Date: {report_date}")
    print(f"   📊 Available Reports: {len(report_files)}")
    print(f"\n   ℹ️  This notebook will automatically parse and visualize")
    print(f"      metrics from the latest performance report.")
else:
    print(f"\n⚠️ No performance reports found in {reports_dir}")
    print(f"   Run the training pipeline first to generate a report.")


✅ All libraries loaded successfully
🎨 Dark mode color palette applied
📅 Presentation Date: December 05, 2025

⚠️ No performance reports found in ../reports
   Run the training pipeline first to generate a report.


---

# 📊 Section 1: Business Problem & Data Overview

---

## 🎯 The Business Challenge

<div style="background:rgb(255, 255, 255); padding: 20px; border-radius: 10px; border-left: 5px solidrgb(0, 0, 0); margin: 15px 0;">

### Problem Statement

**Late deliveries cost e-commerce companies millions annually:**
- 📉 Customer churn from unexpected delays
- 💸 Refunds and compensation costs
- 📞 Increased customer service load
- ⭐ Negative reviews and reputation damage

</div>

### 💡 Our Solution

**Build a Machine Learning system to predict late deliveries BEFORE they happen**, enabling:
- Proactive customer communication
- Shipping upgrades for at-risk orders
- Operational resource optimization


In [45]:
# ============================================================
# 📂 LOAD RAW DATA
# ============================================================
from src.data.data_manager import load_raw

# Load raw dataset
df_raw = load_raw()

# Standardize column names
df_raw.columns = (df_raw.columns
                  .str.strip()
                  .str.lower()
                  .str.replace(' ', '_')
                  .str.replace('(', '')
                  .str.replace(')', ''))

print(f"\n{'='*60}")
print("📊 DATASET OVERVIEW")
print(f"{'='*60}")
print(f"\n📦 Total Records: {df_raw.shape[0]:,} e-commerce orders")
print(f"📋 Total Features: {df_raw.shape[1]} columns")
print(f"📅 Memory Usage: {df_raw.memory_usage(deep=True).sum() / 1024**2:.1f} MB")


📂 Loading raw file: /Users/unclesam/Projects/supply-chain-ml-project/data/raw/DataCoSupplyChainDataset.csv
⚠️ UTF-8 decode failed. Retrying with Latin-1...

📊 DATASET OVERVIEW

📦 Total Records: 180,519 e-commerce orders
📋 Total Features: 53 columns
📅 Memory Usage: 332.6 MB


In [46]:
# ============================================================
# 📊 DATASET SCHEMA VISUALIZATION (Dark Mode)
# ============================================================

# Data types breakdown
dtype_counts = df_raw.dtypes.value_counts()
missing_pct = (df_raw.isnull().sum() / len(df_raw) * 100).sort_values(ascending=False)

# Create overview visualization
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "pie"}, {"type": "bar"}]],
    subplot_titles=('<b>Data Types Distribution</b>', '<b>Missing Values Analysis</b>')
)

# Data types pie chart (dark mode colors)
fig.add_trace(
    go.Pie(
        labels=[str(d) for d in dtype_counts.index],
        values=dtype_counts.values,
        hole=0.4,
        marker_colors=CATEGORICAL_COLORS[:len(dtype_counts)],
        textinfo='percent+label',
        textfont=dict(size=12, color=DARK_COLORS['text'])
    ),
    row=1, col=1
)

# Missing values bar chart (dark mode)
missing_cols = missing_pct[missing_pct > 0]
if len(missing_cols) > 0:
    fig.add_trace(
        go.Bar(
            x=missing_cols.values[:10],
            y=missing_cols.index[:10],
            orientation='h',
            marker_color=DARK_COLORS['coral'],
            text=[f"{v:.1f}%" for v in missing_cols.values[:10]],
            textposition='outside',
            textfont=dict(color=DARK_COLORS['text'])
        ),
        row=1, col=2
    )

fig.update_layout(
    height=400,
    title_text='<b>Dataset Structure Overview</b>',
    showlegend=False,
    paper_bgcolor=DARK_COLORS['background'],
    plot_bgcolor=DARK_COLORS['card']
)
fig.update_xaxes(title_text="Missing %", row=1, col=2, gridcolor=DARK_COLORS['surface'])
fig.update_yaxes(gridcolor=DARK_COLORS['surface'])
fig.show()

# Summary statistics
print(f"\n📋 DATA QUALITY SUMMARY:")
print(f"   • Total missing cells: {df_raw.isnull().sum().sum():,} ({df_raw.isnull().sum().sum()/(len(df_raw)*len(df_raw.columns))*100:.2f}%)")
print(f"   • Columns with missing data: {(df_raw.isnull().sum() > 0).sum()} out of {len(df_raw.columns)}")
print(f"   • Numeric features: {len(df_raw.select_dtypes(include=[np.number]).columns)}")
print(f"   • Categorical features: {len(df_raw.select_dtypes(include=['object']).columns)}")



📋 DATA QUALITY SUMMARY:
   • Total missing cells: 336,209 (3.51%)
   • Columns with missing data: 4 out of 53
   • Numeric features: 29
   • Categorical features: 24


In [47]:
# ============================================================
# 🎯 TARGET VARIABLE ANALYSIS (Dark Mode)
# ============================================================

target_col = 'late_delivery_risk'
target_dist = df_raw[target_col].value_counts()
target_pct = df_raw[target_col].value_counts(normalize=True) * 100

# Also get delivery status distribution
status_dist = df_raw['delivery_status'].value_counts()

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "pie"}, {"type": "bar"}]],
    subplot_titles=('<b>Late Delivery Risk (Target)</b>', '<b>Delivery Status Breakdown</b>')
)

# Target pie chart (dark mode colors)
fig.add_trace(
    go.Pie(
        labels=['Late Delivery', 'On-Time'],
        values=[target_dist.get(1, 0), target_dist.get(0, 0)],
        hole=0.4,
        marker_colors=[STATUS_COLORS['late'], STATUS_COLORS['ontime']],
        textinfo='percent+label',
        textfont=dict(size=14, color=DARK_COLORS['text'])
    ),
    row=1, col=1
)

# Status bar chart (dark mode)
status_colors_map = {
    'Late delivery': STATUS_COLORS['late'],
    'Advance shipping': STATUS_COLORS['ontime'],
    'Shipping on time': DARK_COLORS['cyan'],
    'Shipping canceled': DARK_COLORS['text_secondary']
}
bar_colors = [status_colors_map.get(s, DARK_COLORS['text_secondary']) for s in status_dist.index]

fig.add_trace(
    go.Bar(
        x=status_dist.index,
        y=status_dist.values,
        marker_color=bar_colors,
        text=[f"{v:,}" for v in status_dist.values],
        textposition='outside',
        textfont=dict(color=DARK_COLORS['text'])
    ),
    row=1, col=2
)

fig.update_layout(
    height=450,
    title_text='<b>Target Variable Distribution: Late Delivery Analysis</b>',
    showlegend=False,
    paper_bgcolor=DARK_COLORS['background'],
    plot_bgcolor=DARK_COLORS['card']
)
fig.update_xaxes(tickangle=45, row=1, col=2, gridcolor=DARK_COLORS['surface'])
fig.update_yaxes(gridcolor=DARK_COLORS['surface'])
fig.show()

print(f"\n🎯 TARGET VARIABLE INSIGHTS:")
print(f"   • Late deliveries: {target_dist.get(1, 0):,} orders ({target_pct.get(1, 0):.1f}%)")
print(f"   • On-time deliveries: {target_dist.get(0, 0):,} orders ({target_pct.get(0, 0):.1f}%)")
print(f"   • Class ratio: 1:{target_dist.get(0, 1)/target_dist.get(1, 1):.2f} (slightly imbalanced)")



🎯 TARGET VARIABLE INSIGHTS:
   • Late deliveries: 98,977 orders (54.8%)
   • On-time deliveries: 81,542 orders (45.2%)
   • Class ratio: 1:0.82 (slightly imbalanced)


---

# 🔍 Section 2: Exploratory Data Analysis (EDA)

---

## Key Analytical Questions

1. **What shipping modes have highest late delivery rates?**
2. **Are there temporal patterns in late deliveries?**
3. **Which customer segments are most affected?**
4. **What markets/regions show highest risk?**


In [48]:
# ============================================================
# 📊 EDA: SHIPPING MODE ANALYSIS
# ============================================================

# Late delivery rate by shipping mode
shipping_analysis = df_raw.groupby('shipping_mode').agg({
    'late_delivery_risk': ['mean', 'count']
}).round(3)
shipping_analysis.columns = ['late_rate', 'order_count']
shipping_analysis = shipping_analysis.sort_values('late_rate', ascending=False)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('<b>Late Delivery Rate by Shipping Mode</b>', '<b>Order Volume by Shipping Mode</b>')
)

# Late rate bar
colors = ['#e74c3c' if r > 0.5 else '#f39c12' if r > 0.4 else '#2ecc71'
          for r in shipping_analysis['late_rate']]

fig.add_trace(
    go.Bar(
        x=shipping_analysis.index,
        y=shipping_analysis['late_rate'] * 100,
        marker_color=colors,
        text=[f"{r*100:.1f}%" for r in shipping_analysis['late_rate']],
        textposition='outside'
    ),
    row=1, col=1
)

# Order count bar
fig.add_trace(
    go.Bar(
        x=shipping_analysis.index,
        y=shipping_analysis['order_count'],
        marker_color='#3498db',
        text=[f"{c:,}" for c in shipping_analysis['order_count']],
        textposition='outside'
    ),
    row=1, col=2
)

fig.update_layout(height=450, showlegend=False,
                  title_text='<b>Shipping Mode Impact on Late Deliveries</b>')
fig.update_yaxes(title_text="Late Rate (%)", row=1, col=1)
fig.update_yaxes(title_text="Order Count", row=1, col=2)
fig.show()

print("\n📦 KEY INSIGHT: Shipping mode selection significantly impacts late delivery risk")
print(f"   • Same Day shipping has lowest late rate")
print(f"   • Standard Class has highest volume but also higher risk")



📦 KEY INSIGHT: Shipping mode selection significantly impacts late delivery risk
   • Same Day shipping has lowest late rate
   • Standard Class has highest volume but also higher risk


In [49]:
# ============================================================
# 📊 EDA: CUSTOMER SEGMENT & MARKET ANALYSIS
# ============================================================

# Customer segment analysis
segment_analysis = df_raw.groupby('customer_segment').agg({
    'late_delivery_risk': ['mean', 'count']
}).round(3)
segment_analysis.columns = ['late_rate', 'order_count']

# Market analysis
market_analysis = df_raw.groupby('market').agg({
    'late_delivery_risk': ['mean', 'count']
}).round(3)
market_analysis.columns = ['late_rate', 'order_count']
market_analysis = market_analysis.sort_values('late_rate', ascending=False)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('<b>Late Rate by Customer Segment</b>', '<b>Late Rate by Market Region</b>')
)

# Customer segment
fig.add_trace(
    go.Bar(
        x=segment_analysis.index,
        y=segment_analysis['late_rate'] * 100,
        marker_color=['#9b59b6', '#3498db', '#e67e22'],
        text=[f"{r*100:.1f}%" for r in segment_analysis['late_rate']],
        textposition='outside'
    ),
    row=1, col=1
)

# Market region
fig.add_trace(
    go.Bar(
        x=market_analysis.index,
        y=market_analysis['late_rate'] * 100,
        marker_color='#16a085',
        text=[f"{r*100:.1f}%" for r in market_analysis['late_rate']],
        textposition='outside'
    ),
    row=1, col=2
)

fig.update_layout(height=400, showlegend=False)
fig.update_yaxes(title_text="Late Rate (%)", range=[0, 70])
fig.show()

print("\n👥 SEGMENT INSIGHTS:")
print(f"   • All customer segments show similar late delivery rates (~55%)")
print(f"   • LATAM and Africa markets show highest risk - logistics challenges")



👥 SEGMENT INSIGHTS:
   • All customer segments show similar late delivery rates (~55%)
   • LATAM and Africa markets show highest risk - logistics challenges


---

# 👥 Customer Clustering Analysis

---

## Understanding Customer Segments with Machine Learning

Customer clustering helps us discover **natural groupings** in our customer base based on their behavior patterns.

### Why Clustering?

- **Identify High-Risk Groups**: Some customer segments may have consistently higher late delivery rates
- **Personalized Interventions**: Different groups may need different shipping strategies
- **Resource Allocation**: Focus attention on segments that drive the most value or risk

### Method: K-Means Clustering

K-Means groups customers by finding the center of each group and assigning customers to their nearest center. Think of it like sorting items into buckets based on similarity.


In [50]:
# ============================================================
# 👥 CUSTOMER CLUSTERING ANALYSIS (Dark Mode)
# ============================================================
from sklearn.metrics import silhouette_score

# Prepare clustering features from raw data
# Select numeric features relevant to customer behavior
cluster_features = ['sales', 'order_item_quantity', 'order_item_discount', 'order_profit_per_order']
available_features = [f for f in cluster_features if f in df_raw.columns]

if len(available_features) >= 2:
    # Prepare data for clustering
    X_cluster = df_raw[available_features].dropna()

    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_cluster)

    # Find optimal K using silhouette score
    silhouette_scores = []
    K_range = range(2, 7)
    for k in K_range:
        kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
        labels = kmeans.fit_predict(X_scaled)
        silhouette_scores.append(silhouette_score(X_scaled, labels) if len(set(labels)) > 1 else 0)

    optimal_k = K_range[np.argmax(silhouette_scores)]

    # Fit final model
    kmeans_final = KMeans(n_clusters=optimal_k, random_state=RANDOM_STATE, n_init=10)
    cluster_labels = kmeans_final.fit_predict(X_scaled)

    # PCA for visualization
    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    X_pca = pca.fit_transform(X_scaled)

    # Create visualization
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('<b>Customer Clusters (PCA Projection)</b>', '<b>Cluster Distribution</b>')
    )

    # Scatter plot
    for i in range(optimal_k):
        mask = cluster_labels == i
        fig.add_trace(
            go.Scatter(
                x=X_pca[mask, 0],
                y=X_pca[mask, 1],
                mode='markers',
                name=f'Cluster {i}',
                marker=dict(size=6, color=CATEGORICAL_COLORS[i], opacity=0.6)
            ),
            row=1, col=1
        )

    # Add centroids
    centroids_pca = pca.transform(kmeans_final.cluster_centers_)
    fig.add_trace(
        go.Scatter(
            x=centroids_pca[:, 0],
            y=centroids_pca[:, 1],
            mode='markers',
            name='Centroids',
            marker=dict(size=15, color=DARK_COLORS['gold'], symbol='x', line=dict(width=2))
        ),
        row=1, col=1
    )

    # Cluster distribution bar
    unique, counts = np.unique(cluster_labels, return_counts=True)
    fig.add_trace(
        go.Bar(
            x=[f'Cluster {c}' for c in unique],
            y=counts,
            marker_color=[CATEGORICAL_COLORS[c] for c in unique],
            text=[f'{c:,}' for c in counts],
            textposition='outside',
            textfont=dict(color=DARK_COLORS['text'])
        ),
        row=1, col=2
    )

    fig.update_layout(
        height=450,
        paper_bgcolor=DARK_COLORS['background'],
        plot_bgcolor=DARK_COLORS['card'],
        showlegend=True,
        legend=dict(x=0.02, y=0.98, bgcolor='rgba(0,0,0,0)')
    )
    fig.update_xaxes(gridcolor=DARK_COLORS['surface'], title_text="PC1", row=1, col=1)
    fig.update_yaxes(gridcolor=DARK_COLORS['surface'], title_text="PC2", row=1, col=1)
    fig.update_xaxes(gridcolor=DARK_COLORS['surface'], row=1, col=2)
    fig.update_yaxes(gridcolor=DARK_COLORS['surface'], title_text="Customers", row=1, col=2)
    fig.show()

    # Print insights
    print(f"\n📊 CUSTOMER CLUSTERING INSIGHTS:")
    print(f"   • Optimal clusters found: {optimal_k}")
    print(f"   • Features used: {', '.join(available_features)}")
    print(f"\n👥 CLUSTER INTERPRETATION (Non-Technical):")

    # Analyze each cluster
    df_clustered = X_cluster.copy()
    df_clustered['cluster'] = cluster_labels

    for cluster_id in range(optimal_k):
        cluster_data = df_clustered[df_clustered['cluster'] == cluster_id]
        avg_sales = cluster_data['sales'].mean() if 'sales' in cluster_data else 0
        avg_qty = cluster_data['order_item_quantity'].mean() if 'order_item_quantity' in cluster_data else 0

        # Simple interpretation
        if avg_sales > df_clustered['sales'].mean() * 1.3:
            segment_desc = "High-Value Customers (Large orders, premium segment)"
        elif avg_sales < df_clustered['sales'].mean() * 0.7:
            segment_desc = "Budget Customers (Smaller orders, price-sensitive)"
        else:
            segment_desc = "Standard Customers (Average purchasing behavior)"

        print(f"   • Cluster {cluster_id}: {len(cluster_data):,} customers - {segment_desc}")
else:
    print("⚠️ Insufficient features for clustering analysis")



📊 CUSTOMER CLUSTERING INSIGHTS:
   • Optimal clusters found: 4
   • Features used: sales, order_item_quantity, order_item_discount, order_profit_per_order

👥 CLUSTER INTERPRETATION (Non-Technical):
   • Cluster 0: 6,486 customers - High-Value Customers (Large orders, premium segment)
   • Cluster 1: 90,934 customers - Budget Customers (Smaller orders, price-sensitive)
   • Cluster 2: 51,921 customers - Standard Customers (Average purchasing behavior)
   • Cluster 3: 31,178 customers - High-Value Customers (Large orders, premium segment)


---

# 🛠️ Section 3: Feature Engineering Deep Dive

---

## ⚠️ Critical: Data Leakage Prevention

<div style="background:rgb(255, 255, 255); padding: 20px; border-radius: 10px; border-left: 5px solid #dc3545; margin: 15px 0;">

### 🚫 Why This Matters

**Data leakage** occurs when features contain information about the target that wouldn't be available at prediction time.

**Excluded Features (contain post-delivery info):**

| Column | Reason | Risk Level |
|--------|--------|------------|
| `late_delivery_risk` | **This IS the target** | 🔴 Critical |
| `delivery_status` | Categorical form of target | 🔴 Critical |
| `days_for_shipping_(real)` | Only known AFTER delivery | 🔴 Critical |
| `shipping_date` | Actual shipping timestamp | 🟡 High |

**Using these features would give 100% accuracy but fail completely in production!**

</div>


In [51]:
# ============================================================
# 🔧 FEATURE ENGINEERING: LOAD PREPROCESSED DATA
# ============================================================
from src.data.preprocess import load_or_preprocess

# Load preprocessed data
df = load_or_preprocess()

print(f"\n{'='*70}")
print("FEATURE ENGINEERING OVERVIEW")
print(f"{'='*70}")
print(f"\nData after preprocessing: {df.shape[0]:,} rows × {df.shape[1]} columns")


📂 Loading latest file: /Users/unclesam/Projects/supply-chain-ml-project/data/interim/cleaned_data_20251205_1250.parquet
✅ Loaded cached preprocessed data from data/interim

FEATURE ENGINEERING OVERVIEW

Data after preprocessing: 180,519 rows × 56 columns


In [52]:
# ============================================================
# 📋 FEATURE ENGINEERING CATEGORIES - DETAILED BREAKDOWN
# ============================================================

# Feature categories visualization
feature_categories = {
    'Temporal Features': {
        'features': ['order_day_of_week', 'order_month', 'order_quarter', 'is_weekend'],
        'description': 'Extracted from ORDER date (not shipping date)',
        'count': 5,
        'color': '#3498db'
    },
    'Customer Features': {
        'features': ['customer_order_count', 'customer_lifetime_value'],
        'description': 'Customer behavior patterns',
        'count': 2,
        'color': '#9b59b6'
    },
    'Product Features': {
        'features': ['product_popularity', 'category_popularity', 'order_value', 'discount_rate'],
        'description': 'Item characteristics and pricing',
        'count': 4,
        'color': '#2ecc71'
    },
    'Shipping Features': {
        'features': ['shipping_urgency', 'scheduled_shipping_days'],
        'description': 'Pre-delivery shipping configuration',
        'count': 2,
        'color': '#e74c3c'
    },
    'Financial Features': {
        'features': ['profit_margin_pct', 'sales_per_item', 'is_high_value'],
        'description': 'Order economics and profitability',
        'count': 3,
        'color': '#f39c12'
    },
    'Encoded Categorical': {
        'features': ['type_encoded', 'category_name_encoded', 'customer_segment_encoded',
                     'market_encoded', 'shipping_mode_encoded', 'order_region_encoded'],
        'description': 'Label-encoded categorical variables',
        'count': 10,
        'color': '#1abc9c'
    }
}

# Create visualization
categories = list(feature_categories.keys())
counts = [feature_categories[c]['count'] for c in categories]
colors = [feature_categories[c]['color'] for c in categories]

fig = go.Figure(data=[go.Pie(
    labels=categories,
    values=counts,
    hole=0.4,
    marker_colors=colors,
    textinfo='value+label',
    textfont_size=11
)])

fig.update_layout(height=450, title_text='<b>Feature Engineering Summary: 26 Total Features</b>')
fig.show()

# Print detailed breakdown
print(f"\n{'='*70}")
print("FEATURE CATEGORIES DETAIL")
print(f"{'='*70}")
total = 0
for cat, info in feature_categories.items():
    print(f"\n📌 {cat} ({info['count']} features)")
    print(f"   Description: {info['description']}")
    print(f"   Features: {', '.join(info['features'][:3])}..." if len(info['features']) > 3 else f"   Features: {', '.join(info['features'])}")
    total += info['count']
print(f"\n{'='*70}")
print(f"TOTAL FEATURES: {total}")
print(f"{'='*70}")



FEATURE CATEGORIES DETAIL

📌 Temporal Features (5 features)
   Description: Extracted from ORDER date (not shipping date)
   Features: order_day_of_week, order_month, order_quarter...

📌 Customer Features (2 features)
   Description: Customer behavior patterns
   Features: customer_order_count, customer_lifetime_value

📌 Product Features (4 features)
   Description: Item characteristics and pricing
   Features: product_popularity, category_popularity, order_value...

📌 Shipping Features (2 features)
   Description: Pre-delivery shipping configuration
   Features: shipping_urgency, scheduled_shipping_days

📌 Financial Features (3 features)
   Description: Order economics and profitability
   Features: profit_margin_pct, sales_per_item, is_high_value

📌 Encoded Categorical (10 features)
   Description: Label-encoded categorical variables
   Features: type_encoded, category_name_encoded, customer_segment_encoded...

TOTAL FEATURES: 26


In [53]:
# ============================================================
# 📊 MISSING VALUE HANDLING STRATEGY
# ============================================================

print(f"\n{'='*70}")
print("MISSING VALUE TREATMENT STRATEGY")
print(f"{'='*70}")

missing_strategy = {
    'High Missing (>50%)': {
        'columns': ['product_description', 'order_zipcode'],
        'action': 'DROPPED - Too sparse to impute meaningfully',
        'rationale': 'Information loss outweighed by data quality gains',
        'color': '#e74c3c'
    },
    'Numeric Columns': {
        'columns': ['customer_zipcode', 'order_item_discount'],
        'action': 'MEDIAN IMPUTATION - Robust to outliers',
        'rationale': 'Preserves central tendency without outlier influence',
        'color': '#3498db'
    },
    'Categorical Columns': {
        'columns': ['customer_lname', 'category_name'],
        'action': 'MODE IMPUTATION - Most frequent value',
        'rationale': 'Maintains category distribution',
        'color': '#2ecc71'
    }
}

for strategy_name, info in missing_strategy.items():
    print(f"\n🔹 {strategy_name}")
    print(f"   Columns: {', '.join(info['columns'])}")
    print(f"   Action: {info['action']}")
    print(f"   Rationale: {info['rationale']}")

print(f"\n\n📊 RESULT: Zero missing values in final feature matrix")



MISSING VALUE TREATMENT STRATEGY

🔹 High Missing (>50%)
   Columns: product_description, order_zipcode
   Action: DROPPED - Too sparse to impute meaningfully
   Rationale: Information loss outweighed by data quality gains

🔹 Numeric Columns
   Columns: customer_zipcode, order_item_discount
   Action: MEDIAN IMPUTATION - Robust to outliers
   Rationale: Preserves central tendency without outlier influence

🔹 Categorical Columns
   Columns: customer_lname, category_name
   Action: MODE IMPUTATION - Most frequent value
   Rationale: Maintains category distribution


📊 RESULT: Zero missing values in final feature matrix


In [54]:
# ============================================================
# 🔧 BUILD FEATURES AND PREPARE DATA
# ============================================================
from src.features.build_features import build_features_pipeline

# Build feature matrix
X, y = build_features_pipeline(df)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"\n{'='*60}")
print("DATA SPLIT SUMMARY")
print(f"{'='*60}")
print(f"Training set: {len(X_train):,} samples ({len(X_train)/len(X)*100:.0f}%)")
print(f"Test set: {len(X_test):,} samples ({len(X_test)/len(X)*100:.0f}%)")
print(f"\nFeatures: {X.shape[1]}")
print(f"Target distribution (train): {y_train.mean()*100:.1f}% late")


FEATURE ENGINEERING PIPELINE
⚠️  LEAKAGE PREVENTION ACTIVE
    Excluded columns: delivery_status, delivery_status_encoded, late_delivery_risk, delivery_days, shipping_date_(dateorders), days_for_shipping_(real)
✅ Temporal features created: day_of_week, month, quarter, is_weekend, days_since_start
✅ Customer features created: order_count, lifetime_value
✅ Product features created: popularity, category_popularity, order_value, discount_rate
✅ Shipping features created: shipping_urgency, scheduled_shipping_days, region_country
✅ Financial features created: profit_margin_pct, sales_per_item, is_high_value
✅ Encoded 10 categorical features
   (Excluded 'delivery_status' to prevent leakage)

✅ Selected 26 features for classification
   (Verified: No leaky features included)

Feature matrix shape: (180519, 26)
Target distribution: {1: 98977, 0: 81542}

DATA SPLIT SUMMARY
Training set: 144,415 samples (80%)
Test set: 36,104 samples (20%)

Features: 26
Target distribution (train): 54.8% late


In [55]:
# ============================================================
# 🤖 DYNAMIC MODELING OVERVIEW (Auto-generated from Report)
# ============================================================
from IPython.display import display, HTML, Markdown

# Generate dynamic content based on parsed report
if report_data and report_data['model_comparison']:
    models = report_data['model_comparison']
    best_model = report_data['best_model']
    best_f1 = report_data['metrics']['test_f1']

    # Find baseline (Logistic Regression) metrics
    baseline = next((m for m in models if 'logistic' in m['model'].lower()), None)
    baseline_f1 = baseline['test_f1'] if baseline else 0.68

    # Count models
    n_models = len(models)

    # Build dynamic table rows
    table_rows = ""
    for m in sorted(models, key=lambda x: x['test_f1'], reverse=True):
        status_icon = "⚠️ Overfitting" if 'Overfit' in str(m.get('status', '')) else "✅ Good Fit"
        roc = f"{m['test_roc_auc']:.3f}" if m.get('test_roc_auc') else "N/A"
        if m['model'] == best_model:
            table_rows += f"| **{m['model']}** | **{m['test_f1']:.3f}** | **{roc}** | {status_icon} |\n"
        else:
            table_rows += f"| {m['model']} | {m['test_f1']:.3f} | {roc} | {status_icon} |\n"

    html_content = f'''
<div style="background: linear-gradient(135deg, #161B22 0%, #21262D 100%); padding: 25px; border-radius: 12px; border-left: 6px solid #00D4FF; margin: 15px 0; color: #F0F6FC; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">

<h2 style="color: #00D4FF; margin-top: 0;">🤖 Section 4: Modeling Approach & Results</h2>
<p style="color: #8B949E; font-size: 0.9em;">Auto-generated from: {report_data['report_name']}</p>

<h3 style="color: #F0F6FC;">🔄 ML Pipeline Flow</h3>

<pre style="background: #0D1117; padding: 15px; border-radius: 8px; color: #58A6FF; overflow-x: auto;">
┌─────────────┐    ┌──────────────┐    ┌───────────────┐    ┌──────────────┐    ┌─────────────┐
│   Baseline  │ →  │  {n_models} Models    │ →  │ Hyperparameter│ →  │   Threshold  │ →  │    Best     │
│  (Logistic) │    │   Trained    │    │    Tuning     │    │ Optimization │    │ ({best_model})  │
└─────────────┘    └──────────────┘    └───────────────┘    └──────────────┘    └─────────────┘
   F1: {baseline_f1:.3f}         Parallel            Optuna           Dynamic          F1: {best_f1:.3f}
</pre>

<h3 style="color: #F0F6FC;">📊 Models Trained (Auto-parsed from Report)</h3>

</div>
'''
    display(HTML(html_content))

    # Display the table using Markdown for better formatting
    table_md = f'''
| Model | Test F1 | Test ROC-AUC | Status |
|-------|---------|--------------|--------|
{table_rows}
'''
    display(Markdown(table_md))

else:
    display(HTML('''
<div style="background: #21262D; padding: 20px; border-radius: 10px; border-left: 5px solid #F85149; color: #F0F6FC;">
<h3>⚠️ No Report Data Available</h3>
<p>Could not parse model comparison data from the performance report.</p>
</div>
'''))


NameError: name 'report_data' is not defined

In [ ]:
# ============================================================
# 🤖 AUTOMATED REPORT PARSER - Reads Latest Performance Report
# ============================================================
import re

model_dir = Path('../models')
reports_dir = Path('../reports')

# ============================================================
# COMPREHENSIVE REPORT PARSER
# ============================================================
def parse_performance_report(report_path):
    """
    Automatically parse the model performance report and extract all metrics.
    Returns a dictionary with all parsed data.
    """
    with open(report_path, 'r') as f:
        content = f.read()

    parsed = {
        'report_name': report_path.name,
        'best_model': None,
        'metrics': {},
        'model_comparison': [],
        'feature_importance': [],
        'threshold_optimization': [],
        'dataset_info': {},
    }

    # === Extract Best Model Info ===
    best_match = re.search(r'\*\*Best Model:\*\* (\w+(?:\s+\w+)*)', content)
    parsed['best_model'] = best_match.group(1) if best_match else 'Unknown'

    # === Extract Executive Summary Metrics ===
    f1_match = re.search(r'\*\*Test F1 Score:\*\* ([\d.]+)', content)
    acc_match = re.search(r'\*\*Test Accuracy:\*\* ([\d.]+)', content)
    roc_match = re.search(r'\*\*Test ROC-AUC:\*\* ([\d.]+)', content)
    threshold_match = re.search(r'\*\*Optimal Threshold:\*\* ([\d.]+)', content)
    status_match = re.search(r'\*\*Fit Status:\*\* (.+)', content)

    parsed['metrics'] = {
        'test_f1': float(f1_match.group(1)) if f1_match else 0,
        'test_accuracy': float(acc_match.group(1)) if acc_match else 0,
        'test_roc_auc': float(roc_match.group(1)) if roc_match else 0,
        'optimal_threshold': float(threshold_match.group(1)) if threshold_match else 0.5,
        'fit_status': status_match.group(1).strip() if status_match else 'Unknown',
    }

    # === Extract Model Comparison Table ===
    # Pattern for the comparison table rows
    comparison_pattern = r'\| (\w+(?:\s+\w+)*) \| ([\d.]+) \| ([\d.]+) \| ([\d.]+) \| ([\d.]+) \| ([\d.]+|N/A) \| (-?[\d.]+) \| (.+?) \|'
    for match in re.finditer(comparison_pattern, content):
        model_name, train_acc, test_acc, train_f1, test_f1, test_roc, acc_gap, status = match.groups()
        if model_name not in ['Model', '-------']:
            parsed['model_comparison'].append({
                'model': model_name.strip(),
                'train_acc': float(train_acc),
                'test_acc': float(test_acc),
                'train_f1': float(train_f1),
                'test_f1': float(test_f1),
                'test_roc_auc': float(test_roc) if test_roc != 'N/A' else None,
                'acc_gap': float(acc_gap),
                'status': status.strip()
            })

    # === Extract Feature Importance ===
    # Pattern: | rank | feature | importance | cumulative |
    importance_pattern = r'\| (\d+) \| ([\w_]+) \| ([\d.]+) \| ([\d.]+)% \|'
    for match in re.finditer(importance_pattern, content):
        rank, feature, importance, cumulative = match.groups()
        parsed['feature_importance'].append({
            'rank': int(rank),
            'feature': feature,
            'importance': float(importance),
            'cumulative': float(cumulative)
        })

    # === Extract Threshold Optimization Results ===
    threshold_pattern = r'\| (\w+(?:\s+\w+)*) \| ([\d.]+) \| ([\d.]+) \| ([\d.]+) \| ([+\-]?[\d.]+)'
    for match in re.finditer(threshold_pattern, content):
        model, threshold, default_f1, opt_f1, improvement = match.groups()
        if model not in ['Model', '-------']:
            parsed['threshold_optimization'].append({
                'model': model.strip(),
                'optimal_threshold': float(threshold),
                'default_f1': float(default_f1),
                'optimized_f1': float(opt_f1),
                'improvement': float(improvement)
            })

    # === Extract Dataset Info ===
    train_samples = re.search(r'Training Set.*?Samples:\*\* ([\d,]+)', content, re.DOTALL)
    test_samples = re.search(r'Test Set.*?Samples:\*\* ([\d,]+)', content, re.DOTALL)
    n_features = re.search(r'\*\*Features:\*\* (\d+)', content)
    total_models = re.search(r'Total models trained: (\d+)', content)

    parsed['dataset_info'] = {
        'train_samples': int(train_samples.group(1).replace(',', '')) if train_samples else 0,
        'test_samples': int(test_samples.group(1).replace(',', '')) if test_samples else 0,
        'n_features': int(n_features.group(1)) if n_features else 0,
        'total_models': int(total_models.group(1)) if total_models else 0,
    }

    return parsed

# ============================================================
# LOAD LATEST REPORT
# ============================================================
report_files = sorted(reports_dir.glob('model_performance_report*.md')) if reports_dir.exists() else []

if report_files:
    report_path = report_files[-1]
    print(f"📊 Auto-loading latest report: {report_path.name}")

    # Parse the report
    report_data = parse_performance_report(report_path)
    report_metrics = report_data['metrics']
    report_metrics['best_model'] = report_data['best_model']

    print(f"\n{'='*70}")
    print("📊 AUTOMATICALLY PARSED METRICS FROM REPORT")
    print(f"{'='*70}")
    print(f"   📄 Report: {report_data['report_name']}")
    print(f"   🏆 Best Model: {report_data['best_model']}")
    print(f"   📈 Test F1 Score: {report_metrics['test_f1']:.4f}")
    print(f"   🎯 Test Accuracy: {report_metrics['test_accuracy']:.4f}")
    print(f"   📊 Test ROC-AUC: {report_metrics['test_roc_auc']:.4f}")
    print(f"   ⚙️ Optimal Threshold: {report_metrics['optimal_threshold']:.4f}")
    print(f"   📋 Fit Status: {report_metrics['fit_status']}")
    print(f"\n   📊 Models parsed: {len(report_data['model_comparison'])}")
    print(f"   🔑 Features parsed: {len(report_data['feature_importance'])}")
else:
    print("⚠️ No performance report found in reports/")
    report_data = None
    report_metrics = {
        'best_model': 'Unknown',
        'test_f1': 0,
        'test_accuracy': 0,
        'test_roc_auc': 0,
        'optimal_threshold': 0.5,
        'fit_status': 'Unknown'
    }

# ============================================================
# LOAD PRE-TRAINED MODEL
# ============================================================
model_files = sorted(model_dir.glob('best_model*.pkl')) if model_dir.exists() else []

if model_files:
    best_model = joblib.load(model_files[-1])
    print(f"\n✅ Loaded model: {model_files[-1].name}")

    if hasattr(best_model, 'feature_names_in_'):
        model_features = list(best_model.feature_names_in_)
        available_features = [f for f in model_features if f in X_test.columns]
        X_test_aligned = X_test[available_features]
        X_train_aligned = X_train[available_features]
    else:
        X_test_aligned = X_test
        X_train_aligned = X_train

    model_name = report_metrics.get('best_model', 'Loaded Model')
else:
    print("⚠️ No model found. Training demo model...")
    best_model = LGBMClassifier(
        n_estimators=200, max_depth=8, learning_rate=0.1,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
    )
    best_model.fit(X_train, y_train)
    X_test_aligned = X_test
    X_train_aligned = X_train
    model_name = "LightGBM (Demo)"


In [ ]:
# ============================================================
# 📊 AUTOMATED MODEL COMPARISON (Dynamic from Report)
# ============================================================

if report_data and report_data['model_comparison']:
    # Build DataFrame from parsed report data
    model_comparison = pd.DataFrame(report_data['model_comparison'])
    model_comparison = model_comparison.rename(columns={
        'model': 'Model',
        'train_f1': 'Train_F1',
        'test_f1': 'Test_F1',
        'test_roc_auc': 'Test_ROC_AUC',
        'acc_gap': 'Acc_Gap',
        'status': 'Status'
    })

    # Sort by Test F1 descending
    model_comparison = model_comparison.sort_values('Test_F1', ascending=False).reset_index(drop=True)

    # Handle missing ROC-AUC values
    model_comparison['Test_ROC_AUC'] = model_comparison['Test_ROC_AUC'].fillna(0)

    # Create comparison visualization
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('<b>Train vs Test F1 Score</b>', '<b>Test ROC-AUC by Model</b>'),
        horizontal_spacing=0.12
    )

    # Colors based on status (dynamically determined)
    colors = [DARK_COLORS['coral'] if 'Overfit' in str(s) else DARK_COLORS['lime']
              for s in model_comparison['Status']]

    # Train vs Test F1 comparison
    fig.add_trace(
        go.Bar(name='Train F1', x=model_comparison['Model'], y=model_comparison['Train_F1'],
               marker_color=DARK_COLORS['cyan'], opacity=0.7),
        row=1, col=1
    )
    fig.add_trace(
        go.Bar(name='Test F1', x=model_comparison['Model'], y=model_comparison['Test_F1'],
               marker_color=colors),
        row=1, col=1
    )

    # ROC-AUC comparison
    fig.add_trace(
        go.Bar(name='Test ROC-AUC', x=model_comparison['Model'], y=model_comparison['Test_ROC_AUC'],
               marker_color=CATEGORICAL_COLORS[:len(model_comparison)],
               text=[f'{v:.3f}' for v in model_comparison['Test_ROC_AUC']],
               textposition='outside',
               textfont=dict(color=DARK_COLORS['text'], size=10)),
        row=1, col=2
    )

    fig.update_layout(
        height=450,
        title_text=f'<b>Model Performance Comparison (Auto-parsed from {report_data["report_name"]})</b>',
        barmode='group',
        paper_bgcolor=DARK_COLORS['background'],
        plot_bgcolor=DARK_COLORS['card'],
        legend=dict(x=0.01, y=0.99, bgcolor='rgba(0,0,0,0)'),
        font=dict(color=DARK_COLORS['text'])
    )
    fig.update_xaxes(tickangle=45, gridcolor=DARK_COLORS['surface'])
    fig.update_yaxes(gridcolor=DARK_COLORS['surface'], range=[0, 1])
    fig.show()

    # Print key insights
    n_overfit = model_comparison['Status'].str.contains('Overfit', case=False, na=False).sum()
    n_good = len(model_comparison) - n_overfit

    print(f"\n📊 MODEL COMPARISON INSIGHTS (Auto-parsed from Report):")
    print(f"   • Total models parsed: {len(model_comparison)}")
    print(f"   • Best model by Test F1: {model_comparison.loc[model_comparison['Test_F1'].idxmax(), 'Model']} ({model_comparison['Test_F1'].max():.4f})")
    if model_comparison['Test_ROC_AUC'].max() > 0:
        print(f"   • Best model by ROC-AUC: {model_comparison.loc[model_comparison['Test_ROC_AUC'].idxmax(), 'Model']} ({model_comparison['Test_ROC_AUC'].max():.4f})")
    print(f"   • Models with good fit: {n_good}")
    print(f"   • Models with overfitting: {n_overfit}")

    if n_overfit > 0:
        print(f"\n⚠️ OVERFITTING NOTE:")
        print(f"   Train-test gap indicates overfitting in some models,")
        print(f"   but they may still achieve best test performance.")
else:
    print("⚠️ No model comparison data found in report - skipping visualization.")


In [ ]:
# ============================================================
# 📊 AUTOMATED FEATURE IMPORTANCE (Dynamic from Report)
# ============================================================

if report_data and report_data['feature_importance']:
    # Build DataFrame from parsed report data
    feature_importance = pd.DataFrame(report_data['feature_importance'])
    feature_importance = feature_importance.rename(columns={
        'feature': 'Feature',
        'importance': 'Importance',
        'cumulative': 'Cumulative'
    })

    # Take top 10 features
    feature_importance = feature_importance.head(10)

    fig = go.Figure()

    fig.add_trace(go.Bar(
        y=feature_importance['Feature'][::-1],
        x=feature_importance['Importance'][::-1],
        orientation='h',
        marker=dict(
            color=feature_importance['Importance'][::-1],
            colorscale=[[0, DARK_COLORS['teal']], [0.5, DARK_COLORS['cyan']], [1, DARK_COLORS['magenta']]],
        ),
        text=[f'{v:.1%}' for v in feature_importance['Importance'][::-1]],
        textposition='outside',
        textfont=dict(color=DARK_COLORS['text'])
    ))

    fig.update_layout(
        title=f'<b>Top {len(feature_importance)} Feature Importance ({report_data["best_model"]} - Auto-parsed)</b>',
        xaxis_title='Importance',
        height=400,
        paper_bgcolor=DARK_COLORS['background'],
        plot_bgcolor=DARK_COLORS['card'],
        font=dict(color=DARK_COLORS['text']),
        xaxis=dict(gridcolor=DARK_COLORS['surface'], tickformat='.0%'),
        yaxis=dict(gridcolor=DARK_COLORS['surface']),
        margin=dict(l=180)
    )
    fig.show()

    # Dynamic insights based on parsed data
    print(f"\n🔑 KEY FEATURE INSIGHTS (Auto-parsed from Report):")
    if len(feature_importance) >= 3:
        print(f"   • Top 3 features explain {feature_importance['Cumulative'].iloc[2]:.1f}% of predictions")
    if len(feature_importance) >= 5:
        print(f"   • Top 5 features account for {feature_importance['Cumulative'].iloc[4]:.1f}% of importance")

    # Identify shipping-related features dynamically
    shipping_features = feature_importance[feature_importance['Feature'].str.contains('ship', case=False)]
    if len(shipping_features) > 0:
        print(f"   • Shipping-related features in top 10: {len(shipping_features)}")

    print(f"\n📦 TOP FEATURES INTERPRETATION:")
    for i, row in feature_importance.head(3).iterrows():
        feature_name = row['Feature'].replace('_', ' ').title()
        print(f"   • {row['Feature']}: {row['Importance']:.1%} importance ({feature_name})")
else:
    print("⚠️ No feature importance data found in report - skipping visualization.")


In [ ]:
# ============================================================
# 📊 MODEL EVALUATION - METRICS FROM REPORT + VERIFICATION
# ============================================================

# Generate predictions from loaded model for verification
y_pred = best_model.predict(X_test_aligned)
y_proba = best_model.predict_proba(X_test_aligned)[:, 1]

# Calculate metrics for verification
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
roc_auc = roc_auc_score(y_test, y_proba)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print(f"\n{'='*70}")
print(f"MODEL PERFORMANCE: {model_name} (Best from Report)")
print(f"{'='*70}")

# Show metrics from report
print(f"\n📊 METRICS FROM TRAINING REPORT:")
print(f"   Test F1 Score:  {report_metrics['test_f1']:.4f}")
print(f"   Test Accuracy:  {report_metrics['test_accuracy']:.4f}")
print(f"   Test ROC-AUC:   {report_metrics['test_roc_auc']:.4f}")
print(f"   Opt. Threshold: {report_metrics['optimal_threshold']:.4f}")

print(f"\n📊 VERIFICATION (Current Data Split):")
print(f"   Accuracy:  {accuracy:.4f} ({accuracy*100:.1f}%)")
print(f"   F1 Score:  {f1:.4f}")
print(f"   Precision: {precision:.4f}")
print(f"   Recall:    {recall:.4f}")
print(f"   ROC-AUC:   {roc_auc:.4f}")

print(f"\n📋 CONFUSION MATRIX (Current):")
print(f"   True Positives (Late caught):     {tp:,}")
print(f"   True Negatives (On-time correct): {tn:,}")
print(f"   False Positives (False alarms):   {fp:,}")
print(f"   False Negatives (Missed late):    {fn:,}")


In [ ]:
# ============================================================
# 📊 VISUALIZATION: MODEL PERFORMANCE DASHBOARD
# ============================================================

fig = make_subplots(
    rows=2, cols=2,
    specs=[[{"type": "heatmap"}, {"type": "scatter"}],
           [{"type": "scatter"}, {"type": "bar"}]],
    subplot_titles=('<b>Confusion Matrix</b>', '<b>ROC Curve</b>',
                   '<b>Precision-Recall Curve</b>', '<b>Metrics Summary</b>')
)

# 1. Confusion Matrix
fig.add_trace(
    go.Heatmap(
        z=cm,
        x=['Pred: On-Time', 'Pred: Late'],
        y=['Actual: On-Time', 'Actual: Late'],
        colorscale='Blues',
        text=cm,
        texttemplate='%{text:,}',
        textfont={'size': 16},
        showscale=False
    ),
    row=1, col=1
)

# 2. ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
fig.add_trace(
    go.Scatter(x=fpr, y=tpr, mode='lines', name=f'ROC (AUC={roc_auc:.3f})',
               line=dict(color='#e74c3c', width=3)),
    row=1, col=2
)
fig.add_trace(
    go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Random',
               line=dict(color='gray', dash='dash')),
    row=1, col=2
)

# 3. Precision-Recall Curve
prec, rec, _ = precision_recall_curve(y_test, y_proba)
fig.add_trace(
    go.Scatter(x=rec, y=prec, mode='lines', name='PR Curve',
               line=dict(color='#2ecc71', width=3)),
    row=2, col=1
)

# 4. Metrics Bar Chart
metrics = ['Accuracy', 'F1 Score', 'Precision', 'Recall', 'ROC-AUC']
values = [accuracy, f1, precision, recall, roc_auc]
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']

fig.add_trace(
    go.Bar(
        x=metrics,
        y=values,
        marker_color=colors,
        text=[f'{v:.3f}' for v in values],
        textposition='outside'
    ),
    row=2, col=2
)

fig.update_layout(
    height=700,
    title_text=f'<b>{model_name} - Performance Dashboard</b>',
    showlegend=True
)
fig.update_xaxes(title_text='False Positive Rate', row=1, col=2)
fig.update_yaxes(title_text='True Positive Rate', row=1, col=2)
fig.update_xaxes(title_text='Recall', row=2, col=1)
fig.update_yaxes(title_text='Precision', row=2, col=1)
fig.update_yaxes(range=[0, 1.1], row=2, col=2)
fig.show()


## 📖 Understanding the Metrics (Non-Technical Guide)

<div style="background: linear-gradient(135deg, #161B22 0%, #21262D 100%); padding: 25px; border-radius: 15px; color: #F0F6FC; margin: 20px 0;">

| Metric | Plain English Explanation | Why It Matters |
|--------|--------------------------|----------------|
| **Accuracy** | How often the model is correct overall. 80% = right 8 out of 10 times. | General measure of correctness |
| **Precision** | When we predict "late delivery", how often are we right? High precision = fewer false alarms. | Avoids wasting resources on orders that aren't actually late |
| **Recall** | Of all actual late deliveries, how many did we catch? High recall = fewer missed late deliveries. | Ensures we don't miss at-risk orders |
| **F1 Score** | Balanced combination of precision and recall. Higher is better (max = 1.0). | Best single metric when both errors matter |
| **ROC-AUC** | How well does the model distinguish late from on-time? 0.5 = random guessing, 1.0 = perfect. | Measures overall discrimination ability |

</div>

### ⚖️ The Precision-Recall Trade-off

In real business terms:
- **High Precision, Lower Recall**: We're very careful about predictions - when we say "late", we're usually right. But we might miss some late deliveries.
- **High Recall, Lower Precision**: We catch most late deliveries, but we might have more false alarms.

**Our goal**: Find the balance that minimizes total business cost.


---

# 🔬 Section 5: SHAP Interpretability & Business Impact

---

## Understanding Model Decisions with SHAP

<div style="background: #d4edda; padding: 20px; border-radius: 10px; border-left: 5px solid #28a745; margin: 15px 0;">

### 💡 What is SHAP?

**SHAP (SHapley Additive exPlanations)** provides:

- **Global Importance**: Which features matter most overall?
- **Local Explanations**: Why was THIS specific order predicted as late?
- **Direction of Impact**: Does higher value increase or decrease risk?

### Business Value

| Use Case | How SHAP Helps |
|----------|----------------|
| Stakeholder Trust | Explain predictions to non-technical teams |
| Model Validation | Verify predictions align with domain knowledge |
| Operational Actions | Identify which factors to target for improvement |

</div>


In [ ]:
# ============================================================
# 🔬 SHAP ANALYSIS SETUP
# ============================================================

print("Initializing SHAP explainer...")

# Sample for faster computation
SHAP_SAMPLE_SIZE = 500
sample_idx = np.random.choice(len(X_test_aligned), min(SHAP_SAMPLE_SIZE, len(X_test_aligned)), replace=False)
X_shap = X_test_aligned.iloc[sample_idx].copy()
y_shap = y_test.iloc[sample_idx].copy()
y_proba_shap = y_proba[sample_idx]

# Create appropriate explainer
model_type = type(best_model).__name__

if 'LGBM' in model_type:
    explainer = shap.TreeExplainer(best_model)
elif 'Stacking' in model_type or 'Voting' in model_type:
    # For ensemble models, use first base estimator if tree-based
    try:
        base_model = best_model.estimators_[0]
        explainer = shap.TreeExplainer(base_model)
    except:
        background = shap.sample(X_train_aligned, 100)
        explainer = shap.KernelExplainer(best_model.predict_proba, background)
else:
    try:
        explainer = shap.TreeExplainer(best_model)
    except:
        background = shap.sample(X_train_aligned, 100)
        explainer = shap.KernelExplainer(best_model.predict_proba, background)

print("Calculating SHAP values...")
shap_values = explainer.shap_values(X_shap)

# Handle binary classification output
if isinstance(shap_values, list):
    shap_values = shap_values[1]  # Positive class (Late)

print(f"\n✅ SHAP values calculated for {len(X_shap)} samples")


In [ ]:
# ============================================================
# 📊 SHAP SUMMARY PLOT - GLOBAL FEATURE IMPORTANCE
# ============================================================

print(f"\n{'='*70}")
print("GLOBAL FEATURE IMPORTANCE (SHAP)")
print(f"{'='*70}")
print("\n🔍 Interpretation Guide:")
print("   • Features at TOP are most important")
print("   • RED dots = high feature values, BLUE = low values")
print("   • RIGHT side = pushes toward 'Late' prediction")
print("   • LEFT side = pushes toward 'On-Time' prediction")

plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_shap, show=False, max_display=15)
plt.title('SHAP Feature Importance - What Drives Late Delivery Predictions?',
          fontsize=14, pad=20, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 📊 SHAP BAR PLOT - AVERAGE IMPORTANCE
# ============================================================

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_shap, plot_type='bar', show=False, max_display=15)
plt.title('Average SHAP Value Magnitude by Feature', fontsize=14, pad=20, fontweight='bold')
plt.tight_layout()
plt.show()

# Create feature importance dataframe
mean_abs_shap = np.abs(shap_values).mean(axis=0)
feature_importance_df = pd.DataFrame({
    'feature': X_shap.columns,
    'importance': mean_abs_shap
}).sort_values('importance', ascending=False)

print("\n📊 TOP 10 FEATURES BY SHAP IMPORTANCE:")
print(feature_importance_df.head(10).to_string(index=False))


In [ ]:
# ============================================================
# 💰 BUSINESS IMPACT ANALYSIS
# ============================================================

print(f"\n{'='*70}")
print("BUSINESS IMPACT ANALYSIS")
print(f"{'='*70}")

# Cost assumptions
COST_PER_LATE = 75  # Customer service, refunds, reputation
COST_PER_INTERVENTION = 15  # Shipping upgrade, communication
REVENUE_SAVED = 50  # Customer retention value

# Scale to annual (full dataset)
scale_factor = len(y) / len(y_test)
total_orders = len(y)
late_orders = y.sum()

# Scenario 1: Without ML
scenario1_cost = late_orders * COST_PER_LATE

# Scenario 2: With ML
caught_late = tp * scale_factor
missed_late = fn * scale_factor
false_alarms = fp * scale_factor

scenario2_late_cost = missed_late * COST_PER_LATE
scenario2_intervention_cost = (caught_late + false_alarms) * COST_PER_INTERVENTION
scenario2_revenue_saved = caught_late * REVENUE_SAVED
scenario2_cost = scenario2_late_cost + scenario2_intervention_cost - scenario2_revenue_saved

net_savings = scenario1_cost - scenario2_cost
roi_pct = (net_savings / scenario1_cost) * 100

print(f"\n💰 COST ASSUMPTIONS:")
print(f"   • Cost per late delivery: ${COST_PER_LATE}")
print(f"   • Cost per intervention: ${COST_PER_INTERVENTION}")
print(f"   • Revenue saved per catch: ${REVENUE_SAVED}")

print(f"\n📊 SCENARIO COMPARISON:")
print(f"\n   WITHOUT ML:")
print(f"   • Late delivery costs: ${scenario1_cost:,.0f}")
print(f"\n   WITH ML:")
print(f"   • Late deliveries caught: {caught_late:,.0f}")
print(f"   • Late deliveries missed: {missed_late:,.0f}")
print(f"   • False alarms: {false_alarms:,.0f}")
print(f"   • Total cost: ${scenario2_cost:,.0f}")

print(f"\n🎯 NET ANNUAL SAVINGS: ${net_savings:,.0f}")
print(f"📈 ROI: {roi_pct:.1f}% cost reduction")


In [ ]:
# ============================================================
# 📊 EXECUTIVE SUMMARY DASHBOARD
# ============================================================

late_rate = y.mean() * 100
recall_pct = tp / (tp + fn) * 100 if (tp + fn) > 0 else 0

fig = make_subplots(
    rows=2, cols=3,
    specs=[[{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}],
           [{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}]],
    subplot_titles=(
        'Total Orders', 'Late Delivery Rate', 'Model Accuracy',
        'Late Deliveries Caught', 'Annual Savings', 'ROI'
    )
)

# Row 1
fig.add_trace(go.Indicator(
    mode="number", value=total_orders,
    number={'valueformat': ',', 'font': {'size': 36, 'color': '#3498db'}}
), row=1, col=1)

fig.add_trace(go.Indicator(
    mode="number", value=late_rate,
    number={'suffix': '%', 'font': {'size': 36, 'color': '#e74c3c'}}
), row=1, col=2)

fig.add_trace(go.Indicator(
    mode="number", value=accuracy*100,
    number={'suffix': '%', 'font': {'size': 36, 'color': '#2ecc71'}}
), row=1, col=3)

# Row 2
fig.add_trace(go.Indicator(
    mode="number", value=recall_pct,
    number={'suffix': '%', 'font': {'size': 36, 'color': '#9b59b6'}}
), row=2, col=1)

fig.add_trace(go.Indicator(
    mode="number", value=net_savings,
    number={'prefix': '$', 'valueformat': ',.0f', 'font': {'size': 36, 'color': '#27ae60'}}
), row=2, col=2)

fig.add_trace(go.Indicator(
    mode="number", value=roi_pct,
    number={'suffix': '%', 'font': {'size': 36, 'color': '#f39c12'}}
), row=2, col=3)

fig.update_layout(
    height=500,
    title='<b>📊 Executive Summary Dashboard</b>',
    paper_bgcolor='#f8f9fa'
)
fig.show()


---

# 📋 Key Takeaways & Recommendations

---

<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 30px; border-radius: 15px; color: white; margin: 20px 0;">

## 🎯 Summary

### Problem
- **55% of orders** experience late delivery, costing millions annually

### Solution
- Built **ML classification system** to predict late deliveries before they happen
- Achieved **~77% F1 Score** with production-ready model

### Business Impact
- **74%+** of late deliveries caught before occurrence
- **$2M+ estimated annual savings**
- Enables proactive customer communication

</div>

---

## 💡 Key Technical Insights

| Aspect | Finding |
|--------|----------|
| **Data Quality** | 4 columns with missing values; 2 dropped (>50% missing) |
| **Feature Engineering** | 26 leakage-free features across 6 categories |
| **Top Predictors** | Shipping mode, scheduled days, geographic location |
| **Best Model** | Stacking Ensemble (LightGBM + XGBoost + CatBoost) |
| **Tuning Method** | Optuna (Bayesian optimization) - 50 trials |

---

## 📈 Recommendations

### Immediate (0-30 days)
1. ✅ Deploy model for real-time scoring
2. ✅ Implement risk-tier based workflows
3. ✅ Train operations team on model outputs

### Short-term (1-3 months)
1. 📊 A/B test intervention strategies
2. 📈 Build monitoring dashboard
3. 🔄 Establish monthly retraining pipeline

### Long-term (3-6 months)
1. 🤖 Automate shipping upgrades for critical orders
2. 🔗 Integrate with carrier APIs
3. 📱 Build customer-facing delivery updates

---

## 🙏 Thank You!

**Questions?**


In [ ]:
# ============================================================
# 📋 FINAL SUMMARY PRINT
# ============================================================

print("\n" + "="*80)
print("🚚 SUPPLY CHAIN LATE DELIVERY PREDICTION - PRESENTATION SUMMARY")
print("="*80)

print(f"""
📊 DATA OVERVIEW
{'─'*60}
   • Total Orders: {total_orders:,}
   • Features: 26 (after engineering)
   • Target: {late_rate:.1f}% late delivery rate

🛠️ FEATURE ENGINEERING
{'─'*60}
   • Missing Values: 4 columns handled (2 dropped, 2 imputed)
   • Categories: Temporal, Customer, Product, Shipping, Financial, Encoded
   • Leakage Prevention: Excluded delivery_status, actual shipping days

🤖 MODEL PERFORMANCE
{'─'*60}
   • Best Model: {model_name}
   • Accuracy: {accuracy:.1%}
   • F1 Score: {f1:.4f}
   • ROC-AUC: {roc_auc:.4f}
   • Improvement over baseline: +{(f1-0.688)/0.688*100:.1f}%

💰 BUSINESS IMPACT
{'─'*60}
   • Late Deliveries Caught: {recall_pct:.0f}%
   • Annual Savings: ${net_savings:,.0f}
   • ROI: {roi_pct:.0f}% cost reduction

🔍 KEY PREDICTORS (SHAP)
{'─'*60}""")

for i, row in feature_importance_df.head(5).iterrows():
    print(f"   {i+1}. {row['feature']}: {row['importance']:.4f}")

print(f"""
{'='*80}
                         END OF PRESENTATION
{'='*80}
""")
